<a href="https://colab.research.google.com/github/office268/ipynb/blob/main/ex_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG — Retrieval Augmented Generation

**RAG** היא טכניקה שמאפשרת למודל שפה לענות על שאלות בהתבסס על **מסמך שאתה מספק** — במקום רק מהידע הפנימי שלו.

### איך זה עובד?
```
PDF ← פיצול לצ'אנקים ← Embeddings ← Vector Store
                                            ↓
שאלה → חיפוש רלוונטי → הקשר + שאלה → Gemini → תשובה
```

| שלב | תפקיד |
|------|--------|
| **PyPDF** | קורא את קובץ ה-PDF |
| **Text Splitter** | מפצל את הטקסט לקטעים קטנים (צ'אנקים) |
| **Embeddings** | הופך כל צ'אנק למספרים שמייצגים את המשמעות שלו |
| **Chroma** | שומר את הצ'אנקים ומאפשר חיפוש לפי משמעות |
| **Gemini** | קורא את הצ'אנקים הרלוונטים ומנסח תשובה |

In [ ]:
# התקנת ספריות
!pip install -q -U langchain langchain-community langchain-google-genai langchain-text-splitters chromadb pypdf

In [ ]:
import os
from google.colab import files, userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# העלאת קובץ PDF
print("העלה קובץ PDF:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"✓ הקובץ '{file_name}' הועלה")

In [ ]:
# קריאת ה-PDF ופיצול לצ'אנקים
documents = PyPDFLoader(file_name).load()

texts = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
).split_documents(documents)

print(f"✓ הקובץ פוצל ל-{len(texts)} צ'אנקים")

In [ ]:
# יצירת בסיס נתונים וקטורי
print("יוצר Embeddings...")
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001")
)
print("✓ Vector Store מוכן")

In [ ]:
# בניית שרשרת ה-RAG
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_template("""
ענה על השאלה הבאה בהתבסס אך ורק על ההקשר הנתון.
אם התשובה לא נמצאת בהקשר, אמור זאת מפורשות.

הקשר:
{context}

שאלה: {question}
""")

rag_chain = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ שרשרת ה-RAG מוכנה")

In [ ]:
# שאל שאלה על המסמך
query = input("מה תרצה לשאול על הקובץ? ")

if query:
    print("\nחושב...")
    print("\n" + "="*40)
    print(rag_chain.invoke(query))